# University Google Drive → Hugging Face: safe archive migration

This notebook is an **audited, user-authorized transfer** from the identified academic Google Drive projects into **private** Hugging Face dataset repositories.

## Safety rules

- **Transfer prepared for Zkri2026.** Once an authorized Hugging Face write token is available as a Colab secret and Google Drive authentication is granted, the notebook uploads eligible files into private dataset repositories. It does not delete Google Drive originals.
- **Nothing is deleted from Google Drive**, even after a successful upload. Deletion requires a separate review.
- Exact duplicate bytes are detected via matching Google Drive **MD5 + size**; same filename alone is insufficient.
- Split `.zip.part001` files are omitted only if their concatenation exactly matches the original ZIP's Drive MD5.
- Sensitive data and credentials are **not uploaded by default**. Do not upload identity-document images, medical datasets, or secrets without checking ownership, licenses, privacy, and permissions.
- ZIP integrity and remote Hugging Face **LFS SHA-256 + size** are checked when available. Items without a remote SHA remain **not fully verified**, and should not be deleted from Drive.
- Hugging Face free users have a documented 100 GB private storage allocation, subject to platform limits and policy. See https://huggingface.co/docs/hub/storage-limits .

## How to run

1. Open this `.ipynb` in Google Colab while signed into the **same Google account** that owns the Drive data.
2. In https://huggingface.co/settings/tokens create a Hugging Face token with **write** rights to your own repos. In Colab **Secrets** (key icon), create a secret named `HF_TOKEN`, paste your token there, and grant notebook access. **Never paste tokens into ChatGPT or a public notebook.**
3. Run the installation cell, then the main code cell with `SYNC_ENABLED = False` to generate `audit_manifest.csv`.
4. Review the file list, especially the `REVIEW_SENSITIVE` and duplicates rows. Change the configured source roots or exclusions if needed.
5. Change `SYNC_ENABLED = True` in the code cell and rerun it. The notebook creates private Hugging Face dataset repositories, uploads nonblocked unique files and checks remote checksums.
6. Download `transfer_results.csv` from the Colab Files panel. Only `VERIFIED_SHA256_AND_SIZE` entries have byte-level remote confirmation. **Do not delete Google Drive originals until every required artifact has a separately verified backup.**

### Known project roots included

- ResNet50 pneumonia: academic submission folder **and** separate experiment workspace.
- Nazih/PolyLoss industrial defects: original full coursework archive.
- Project Management Lab06 document quality: results, coursework and original dataset ZIP (ZIP **blocked pending sensitive-data review**).
- Summer Practices 2025 and 2026: original project folders.

The notebook preserves original relative paths within `models/`, `datasets/`, `code/`, `results/`, `documents/`, `other/`. It does **not** rewrite or certify scientific conclusions or fix corrupted models automatically. Invalid ZIP files are reported instead of silently 'repaired'.


In [ ]:
%pip -q install 'huggingface_hub==0.36.0' 'google-auth==2.49.0' 'google-api-python-client'


### Transfer to Zkri2026
This notebook is configured to transfer eligible files as soon as Colab authenticates to Google Drive and detects a Hugging Face `HF_TOKEN` with write access for account `Zkri2026`. No passwords or tokens should be entered into public GitHub files. Sensitive/clinical datasets and credentials remain excluded pending review.


In [ ]:
"""Google Drive → private Hugging Face archives. Run in Google Colab.

Safety defaults:
- Audit only, not upload.
- Never remove or rename source files on Google Drive.
- Deduplicate only byte-identical Google Drive files with matching MD5 and size.
- Skip sensitive datasets/documents until separately authorized.
- Verify a remote LFS SHA256+size where metadata is available.
"""

from __future__ import annotations

import csv
import hashlib
import os
import re
import tempfile
import zipfile
from collections import defaultdict
from pathlib import Path
from typing import Any

# 1. These are Drive IDs previously verified against the user's connected Drive.
# Each root gets a unique canonical source label. A Drive ID encountered twice
# is counted and uploaded once per project.
SOURCES = {
    "university-resnet50-pneumonia-archive": [
        ("academic-coursework", "1Ch62wticPBABtiaH7fCbIReTq1BXuVU4", "folder"),
        ("experiment-workspace", "1cQeYJ6VCOZc-zztC5TQa8AkZ-8rkfDbM", "folder"),
    ],
    "university-industrial-defects-polyloss-archive": [
        ("coursework-and-evidence", "1OI3zrAtmA9ZfwH3l_DB3IiwFBK2P1_ny", "folder"),
    ],
    "university-lab06-document-quality-archive": [
        ("lab06-experiments", "1ja5uljMIcMSNLd41scx2HHV5B0SLallu", "folder"),
        ("lab06-reports-and-tasks", "1id2pTjvu1SnNwshRJjRc-Teno5AGGBIg", "folder"),
        ("original-dataset-zip", "1QNHsbtpWcmeo0asG2Mrrruwy5w2lmobB", "file"),
    ],
    "university-summer-practice-2025-archive": [
        ("digit-recognition-project", "1LmOjZmJxwdVzN7xMAxLaJYQOd2HACjkn", "folder"),
    ],
    "university-summer-practice-2026-archive": [
        ("equipment-booking-project", "1u3kMCTVIab07B3mpLclCUOfFWEK7Wbk_", "folder"),
    ],
}

# User has explicitly authorized transfer of non-sensitive, distinct archive files.
# The actual upload still requires user-authenticated Google Drive and a HF write token.
SYNC_ENABLED = True
EXPECTED_HF_USERNAME = "Zkri2026"
# Source document photos and clinical datasets need a separate licensing/privacy decision.
ALLOW_SENSITIVE_DATA = False
# If True, CRC-check every downloaded .zip before upload (may be CPU-intensive).
TEST_ZIP_CONTENTS = True
# During the audit, validate whether split .zip.part001... files reproduce a full ZIP.
CHECK_SPLIT_ARCHIVES = True
# If true, require the Hugging Face LFS SHA256, not just file-size confirmation.
REQUIRE_REMOTE_SHA256 = True
# Optional subset, e.g. ['university-resnet50-pneumonia-archive']; None = all.
ONLY_PROJECTS = None

NATIVE_EXPORTS = {
    "application/vnd.google-apps.document": ("application/pdf", ".pdf"),
    "application/vnd.google-apps.spreadsheet": (
        "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet", ".xlsx"),
    "application/vnd.google-apps.presentation": (
        "application/vnd.openxmlformats-officedocument.presentationml.presentation", ".pptx"),
}
ALLOWED_REPO_CHARS = re.compile(r"[^a-zA-Z0-9._-]+")


def safe_filename(s: str) -> str:
    s = s.strip().replace("\\", "_").replace("/", "_")
    s = re.sub(r"[\x00-\x1f]", "_", s)
    return s[:180] or "unnamed"


def file_category(path: str, name: str) -> str:
    lower = ("/" + path + "/" + name).lower()
    ext = Path(name).suffix.lower()
    if ext in (".pt", ".pth", ".ckpt", ".keras", ".onnx", ".h5", ".safetensors"):
        return "models"
    if any(x in lower for x in ("/dataset", "/raw/", "/images/", "/data/")) or \
            "dataset" in lower or ext in (".parquet", ".tfrecord"):
        return "datasets"
    if ext in (".py", ".ipynb", ".sh", ".ps1", ".cpp", ".h", ".ts", ".tsx", ".kt", ".java"):
        return "code"
    if any(x in lower for x in ("/results", "/outputs", "/experiments", "checkpoint")) \
            or ext in (".json", ".jsonl", ".csv", ".png", ".jpg"):
        return "results"
    if ext in (".pdf", ".doc", ".docx", ".pptx", ".md", ".txt", ".xlsx"):
        return "documents"
    return "other"


def sensitivity(project: str, path: str, name: str) -> str:
    lower = (path + "/" + name).lower()
    base = name.lower()
    # Secrets always take precedence, including when located inside a dataset folder.
    if base in ("kaggle.json", ".env", "secrets.json", "credentials.json") or \
            base.startswith(".env.") or base.endswith((".pem", ".key", ".p12", ".pfx", ".keystore")):
        return "NEVER_UPLOAD_SECRET"
    # Source images from passport, SNILS, INN may contain identifying information,
    # even when synthetically augmented. The entire Lab06 dataset is gated.
    if project == "university-lab06-document-quality-archive":
        if any(k in lower for k in ("dataset", "passport", "snils", "inn", "raw/", "yolo_data")):
            return "REVIEW_PERSONAL_DOCUMENT_IMAGES"
    if project == "university-resnet50-pneumonia-archive":
        if any(k in lower for k in ("/dataset", "/data/", "chest_xray", "patient", "clinical")):
            return "REVIEW_MEDICAL_DATA_LICENSE"
    return ""


def sha256_file(filepath: Path) -> str:
    h = hashlib.sha256()
    with filepath.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def connect_drive():
    try:
        from google.colab import auth  # type: ignore
        auth.authenticate_user()
        import google.auth
        from googleapiclient.discovery import build
    except ImportError as e:
        raise RuntimeError("Run this script in Google Colab with your Google account.") from e
    creds, _ = google.auth.default()
    return build("drive", "v3", credentials=creds, cache_discovery=False)


def drive_meta(service, file_id: str) -> dict[str, Any]:
    return service.files().get(fileId=file_id, fields=(
        "id,name,mimeType,size,md5Checksum,parents,webViewLink,description,createdTime,modifiedTime,trashed"
    ), supportsAllDrives=True).execute()


def direct_children(service, folder_id: str):
    page = None
    while True:
        r = service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum,parents,webViewLink,description,createdTime,modifiedTime)",
            pageSize=1000,
            pageToken=page,
            supportsAllDrives=True,
            includeItemsFromAllDrives=True,
        ).execute()
        yield from r.get("files", [])
        page = r.get("nextPageToken")
        if not page:
            break


def collect_files(service, project: str, roots):
    pending = []
    for label, file_id, kind in roots:
        if kind == "folder":
            pending.append((file_id, label, ""))
        else:
            f = drive_meta(service, file_id)
            yield project, label, f.get("name", file_id), f
    visited_folders = set()
    while pending:
        folder_id, label, relpath = pending.pop()
        if folder_id in visited_folders:
            continue
        visited_folders.add(folder_id)
        for f in direct_children(service, folder_id):
            name = safe_filename(f.get("name", f["id"]))
            child_path = f"{relpath}/{name}".strip("/")
            if f.get("mimeType") == "application/vnd.google-apps.folder":
                pending.append((f["id"], label, child_path))
            elif f.get("mimeType") == "application/vnd.google-apps.shortcut":
                # Shortcuts have no bytes of their own. Do not follow arbitrary targets.
                continue
            else:
                yield project, label, child_path, f


def target_path(label: str, original_path: str, mime: str) -> str:
    path = Path(original_path)
    name = safe_filename(path.name)
    suffix = NATIVE_EXPORTS.get(mime)
    if suffix:
        name += suffix[1]
    category = file_category(original_path, name)
    ancestors = [safe_filename(x) for x in path.parts[:-1]]
    return "/".join([category, safe_filename(label), *ancestors, name])


def plan(service):
    records = []
    for project, roots in SOURCES.items():
        if ONLY_PROJECTS and project not in ONLY_PROJECTS:
            continue
        hashes = {}
        ids = {}
        destinations = set()
        for _, label, path, f in collect_files(service, project, roots):
            fid = f["id"]
            size = int(f.get("size") or 0)
            md5 = f.get("md5Checksum") or ""
            mime = f.get("mimeType", "")
            dst = target_path(label, path, mime)
            danger = sensitivity(project, path, f.get("name", ""))
            status = "UPLOAD"
            duplicate_of = ""
            if danger == "NEVER_UPLOAD_SECRET":
                status = "SKIP_SECRET"
            elif danger and not ALLOW_SENSITIVE_DATA:
                status = "REVIEW_SENSITIVE"
            elif fid in ids:
                status, duplicate_of = "SKIP_SAME_FILE", ids[fid]
            elif md5 and (md5, size) in hashes:
                status, duplicate_of = "SKIP_EXACT_DUPLICATE", hashes[(md5, size)]
            if status == "UPLOAD":
                if dst in destinations:
                    suffix = (md5 or fid).replace("/", "")[:10]
                    dst = f"{dst}__{suffix}"
                destinations.add(dst)
                if md5:
                    hashes[(md5, size)] = dst
                ids[fid] = dst
            records.append({
                "project": project,
                "source_label": label,
                "source_path": path,
                "parent_id": (f.get("parents") or [""])[0],
                "file_id": fid,
                "mime_type": mime,
                "size_bytes": size,
                "drive_md5": md5,
                "destination_path": dst,
                "sensitivity_flag": danger,
                "plan": status,
                "duplicate_of": duplicate_of,
                "result": "NOT_STARTED",
                "sha256": "",
                "remote_sha256": "",
            })
    return records


def eliminate_verified_split_zip_copies(service, records):
    """Exact byte-level comparison: assembled .zip.partNNN vs complete .zip.

    Never assume matching names/sizes are sufficient. Download parts, stream
    their concatenation through MD5, compare against Drive's full ZIP MD5.
    Only if a verified whole file matches do we omit these redundant pieces.
    """
    if not CHECK_SPLIT_ARCHIVES:
        return
    groups = defaultdict(list)
    complete = {}
    for r in records:
        k = (r["project"], r["parent_id"], str(Path(r["source_path"]).parent))
        basename = Path(r["source_path"]).name
        match = re.match(r"^(.*\.zip)\.part(\d+)$", basename, flags=re.I)
        if match:
            groups[k + (match.group(1),)].append((int(match.group(2)), r))
        elif basename.lower().endswith(".zip"):
            complete[k + (basename,)] = r
    for key, parts in groups.items():
        full = complete.get(key)
        if not full or not full.get("drive_md5"):
            continue
        parts.sort(key=lambda x: x[0])
        numbers = [i for i, _ in parts]
        if numbers != list(range(1, len(numbers)+1)):
            print("SPLIT ARCHIVE INCOMPLETE:", key[-1], numbers)
            continue
        total = sum(r["size_bytes"] for _, r in parts)
        if total != full["size_bytes"]:
            print("SPLIT ARCHIVE SIZE MISMATCH:", key[-1])
            continue
        digest = hashlib.md5()
        try:
            for _, part in parts:
                with tempfile.TemporaryDirectory() as t:
                    f = Path(t) / "piece.bin"
                    download_drive_file(service, part, f)
                    with f.open("rb") as stream:
                        for block in iter(lambda: stream.read(8*1024*1024), b""):
                            digest.update(block)
        except Exception as e:
            print("Cannot verify split archive:", key[-1], e)
            continue
        if digest.hexdigest().lower() == full["drive_md5"].lower():
            print("VERIFIED REDUNDANT SPLIT ARCHIVE:", key[-1])
            for _, part in parts:
                if part["plan"] == "UPLOAD":
                    part["plan"] = "SKIP_VERIFIED_SPLIT_ARCHIVE"
                    part["duplicate_of"] = full["destination_path"]
        else:
            print("Split archive differs from complete ZIP; preserving all:", key[-1])


def write_csv(records, file_path: str):
    if not records:
        return
    with open(file_path, "w", newline="", encoding="utf-8-sig") as o:
        writer = csv.DictWriter(o, fieldnames=records[0].keys())
        writer.writeheader()
        writer.writerows(records)


def download_drive_file(service, record: dict, out: Path):
    from googleapiclient.http import MediaIoBaseDownload
    mime = record["mime_type"]
    if mime in NATIVE_EXPORTS:
        export_mime, _ = NATIVE_EXPORTS[mime]
        request = service.files().export_media(fileId=record["file_id"], mimeType=export_mime)
    else:
        request = service.files().get_media(fileId=record["file_id"], supportsAllDrives=True)
    with out.open("wb") as writer:
        loader = MediaIoBaseDownload(writer, request, chunksize=8 * 1024 * 1024)
        done = False
        while not done:
            _, done = loader.next_chunk(num_retries=4)


def verify_zip(path: Path):
    if path.suffix.lower() != ".zip":
        return
    with zipfile.ZipFile(path) as z:
        if TEST_ZIP_CONTENTS:
            corrupt = z.testzip()
            if corrupt:
                raise ValueError(f"ZIP CRC FAILED at {corrupt}")


def verify_remote(api, repo_id: str, target: str, expected_sha: str, expected_size: int):
    objs = api.get_paths_info(repo_id=repo_id, paths=[target], repo_type="dataset", expand=True)
    obj = objs[0] if objs else None
    if obj is None:
        raise ValueError("Remote path is missing after upload")
    if getattr(obj, "size", None) != expected_size:
        raise ValueError("Remote file size does not match local upload")
    lfs = getattr(obj, "lfs", None)
    remote_sha = lfs.get("sha256") if isinstance(lfs, dict) else getattr(lfs, "sha256", None)
    if remote_sha:
        if remote_sha.lower() != expected_sha.lower():
            raise ValueError("Remote LFS SHA-256 does not match local file")
        return "VERIFIED_SHA256_AND_SIZE", remote_sha
    if REQUIRE_REMOTE_SHA256:
        return "UPLOADED_SIZE_ONLY_NOT_FULLY_VERIFIED", ""
    return "VERIFIED_SIZE_ONLY", ""


def upload_records(service, records, api, username: str):
    """Use the already authenticated Hugging Face client from main().

    Colab Secrets must be fetched only once per run. Calling userdata.get()
    again after a lengthy Drive audit can time out, even when login succeeded.
    """
    if username.lower() != EXPECTED_HF_USERNAME.lower():
        raise RuntimeError("Hugging Face account mismatch; refusing transfer")
    repo_ids = {}
    for project in sorted({r["project"] for r in records if r["plan"] == "UPLOAD"}):
        repo_id = f"{username}/{project}"
        api.create_repo(repo_id=repo_id, repo_type="dataset", private=True, exist_ok=True)
        info = api.repo_info(repo_id=repo_id, repo_type="dataset")
        if not info.private:
            raise RuntimeError(f"Refusing to upload to PUBLIC repo: {repo_id}")
        repo_ids[project] = repo_id

    for i, r in enumerate(records, 1):
        if r["plan"] != "UPLOAD":
            r["result"] = r["plan"]
            continue
        repo_id = repo_ids[r["project"]]
        temp_suffix = Path(r["destination_path"]).suffix or ".bin"
        with tempfile.TemporaryDirectory(prefix="gd_hf_") as t:
            filepath = Path(t) / ("file" + temp_suffix)
            try:
                print(f"[{i}/{len(records)}] Download: {r['project']} / {r['source_path']}")
                download_drive_file(service, r, filepath)
                expected = r["size_bytes"]
                actual = filepath.stat().st_size
                if expected and r["mime_type"] not in NATIVE_EXPORTS and actual != expected:
                    raise ValueError(f"Incomplete Drive transfer: {actual} != {expected}")
                verify_zip(filepath)
                sha = sha256_file(filepath)
                r["sha256"] = sha
                api.upload_file(
                    path_or_fileobj=str(filepath),
                    repo_id=repo_id,
                    repo_type="dataset",
                    path_in_repo=r["destination_path"],
                    commit_message="Archive verified Drive file",
                )
                status, remote_sha = verify_remote(api, repo_id, r["destination_path"], sha, actual)
                r["result"] = status
                r["remote_sha256"] = remote_sha
                print(f"  {status}: {actual} bytes")
            except Exception as e:
                r["result"] = "ERROR: " + str(e)[:250]
                print("  ERROR:", e)
        write_csv(records, "transfer_results.csv")
    return repo_ids


def main():
    api = None
    actual_user = None
    if SYNC_ENABLED:
        # Fail early if credentials are missing, rather than scanning for minutes.
        from google.colab import userdata
        from huggingface_hub import HfApi
        token = userdata.get("HF_TOKEN")
        if not token:
            raise RuntimeError(
                "Hugging Face authorization is required. Add an HF_TOKEN secret "
                "with write permission to Colab Secrets and enable notebook access. "
                "Never place this token in GitHub or in chat."
            )
        api = HfApi(token=token)
        actual_user = api.whoami()["name"]
        if actual_user.lower() != EXPECTED_HF_USERNAME.lower():
            raise RuntimeError(
                f"Connected Hugging Face account {actual_user!r} does not match "
                f"expected account {EXPECTED_HF_USERNAME!r}."
            )
        print("Authenticated destination:", EXPECTED_HF_USERNAME)
    service = connect_drive()
    print("Scanning source folders, inspecting size and md5Checksum for exact duplicates...")
    records = plan(service)
    eliminate_verified_split_zip_copies(service, records)
    write_csv(records, "audit_manifest.csv")
    counts = defaultdict(lambda: [0, 0])
    for r in records:
        counts[(r["project"], r["plan"])][0] += 1
        counts[(r["project"], r["plan"])][1] += r["size_bytes"]
    for (project, status), (n, size) in sorted(counts.items()):
        print(f"{project:52s} {status:26s} {n:5d} files / {size/1e9:8.3f} GB")
    print("AUDIT COMPLETE: audit_manifest.csv")
    if not SYNC_ENABLED:
        print("DRY RUN ONLY. Set SYNC_ENABLED=True after reviewing the CSV. Nothing was uploaded or deleted.")
        return
    repos = upload_records(service, records, api, actual_user)
    write_csv(records, "transfer_results.csv")
    print("PRIVATE REPOSITORIES:", repos)
    print("Google Drive originals have NOT been deleted.")
    print("Only files with VERIFIED_SHA256_AND_SIZE qualify for a later manual deletion review.")


if __name__ == "__main__":
    main()


In [ ]:
# Optional: download the manifest to your computer once the audit has finished.
from google.colab import files
files.download("transfer_results.csv" if SYNC_ENABLED else "audit_manifest.csv")
